# PAI 応用編1 最終課題：四足ロボット Go2 の八の字歩行（Genesis × 強化学習）

**上から順に実行するだけ**で、学習 → 八の字歩行の評価 → 提出用動画の作成 まで終わります。

| Step | 内容 | 目安（T4） |
|---|---|---|
| 0 | GPU確認・インストール・描画(EGL)設定 | 3〜5分 |
| 1 | Google Drive をマウントし、コードを書き出す | 1分 |
| 2 | 強化学習（PPO, 4096並列環境, 500 iter） | 20〜40分 |
| 3 | 八の字歩行の評価＋2カメラ録画 | 5〜10分 |
| 4 | 結果グラフと提出用動画（タイトル＋2画面＋結果）を作成 | 1分 |

**しくみ（階層型制御）**
- 下位：PPO で学習した歩行方策。入力は体の状態＋速度コマンド [前進速度, 横速度, 旋回角速度]、出力は12関節の目標角度。
- 上位：八の字（レムニスケート）の経路を Pure Pursuit で追従し、毎ステップ速度コマンドを生成。

ベース：Genesis 公式 `examples/locomotion`（go2_env / go2_train / go2_eval, Genesis 1.4.5）。
参考：[1] tototototototo7「google colabを利用して自前GPUなしでGenesisの強化学習を試す」、[2] npaka「Genesis 入門 (11)」

> ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから実行してください。

## Step 0. 環境準備

In [ ]:
!nvidia-smi
!python --version

In [ ]:
# Genesis 1.4.5 と rsl-rl-lib 5.x（Genesis 1.4.5 の locomotion 例は rsl-rl-lib>=5 が必須。参考記事の v1.0.2 は旧版用）
!pip install -q genesis-world==1.4.5 rsl-rl-lib==5.5.1 tensorboard

ヘッドレス描画（EGL）の設定。Genesis は学習時にも描画系を初期化するので、**学習の前に**実行します。
NVIDIA の EGL を登録し、念のため Mesa の EGL も入れておきます（GPU の EGL が使えない場合の保険）。

In [ ]:
import os, subprocess
NVIDIA_ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
subprocess.run("apt-get -qq install -y libegl1 libgl1-mesa-dri > /dev/null", shell=True)
os.environ["PYOPENGL_PLATFORM"] = "egl"
print("EGL setup done")

## Step 1. Google Drive にコードを書き出す
Colab が切断されても学習ログ・モデルが残るよう、作業フォルダを Drive に置きます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = "/content/drive/MyDrive/pai_fig8"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

### 環境（fig8_env.py）
公式 `go2_env.py` からの変更点：外部から速度コマンドを与える `set_commands()`、録画用カメラ、旋回を安定させる報酬（`ang_vel_xy`, `orientation`, `dof_acc`）、直進/旋回のみのコマンドを混ぜるサンプリング。

In [ ]:
%%writefile fig8_env.py
import math

import torch
from tensordict import TensorDict

import genesis as gs
from genesis.utils.geom import quat_to_xyz, transform_by_quat, inv_quat, transform_quat_by_quat


def gs_rand(lower, upper, batch_shape):
    assert lower.shape == upper.shape
    return (upper - lower) * torch.rand(size=(*batch_shape, *lower.shape), dtype=gs.tc_float, device=gs.device) + lower


class Fig8Env:
    """Go2 velocity-tracking environment (based on Genesis examples/locomotion/go2_env.py).

    Changes from the original example:
    - omnidirectional commands (forward speed + yaw rate) so the policy can turn both ways
    - extra reward terms that keep the body stable while turning
    - commands can be set from outside (used by the figure-eight path follower at evaluation time)
    - optional cameras for headless video recording (Google Colab)
    """

    def __init__(self, num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg, show_viewer=False, cameras=None):
        self.num_envs: int = num_envs
        self.num_actions = env_cfg["num_actions"]
        self.cfg = env_cfg
        self.num_commands = command_cfg["num_commands"]
        self.device = gs.device

        self.simulate_action_latency = env_cfg.get("simulate_action_latency", True)  # there is a 1 step latency on real robot
        self.dt = 0.02  # control frequency on real robot is 50hz
        self.max_episode_length = math.ceil(env_cfg["episode_length_s"] / self.dt)

        self.env_cfg = env_cfg
        self.obs_cfg = obs_cfg
        self.reward_cfg = reward_cfg
        self.command_cfg = command_cfg

        self.obs_scales: dict[str, float] = obs_cfg["obs_scales"]
        self.reward_scales: dict[str, float] = reward_cfg["reward_scales"]

        self.scene = gs.Scene(
            sim_options=gs.options.SimOptions(
                dt=self.dt,
                substeps=2,
            ),
            rigid_options=gs.options.RigidOptions(
                enable_self_collision=False,
                # For this locomotion policy, there are usually no more than 20 collision pairs. Setting a low value
                # can save memory. Violating this condition will raise an exception.
                max_collision_pairs=20,
                tolerance=1e-5,
            ),
            vis_options=gs.options.VisOptions(
                rendered_envs_idx=[0],
            ),
            viewer_options=gs.options.ViewerOptions(
                camera_pos=(2.0, 0.0, 2.5),
                camera_lookat=(0.0, 0.0, 0.5),
                camera_fov=40,
            ),
            show_viewer=show_viewer,
        )
        self.external_commands = False  # True: commands are given by set_commands() instead of random sampling

        # add plain
        self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/plane/plane.urdf",
                fixed=True,
            )
        )

        self.robot = self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/go2/urdf/go2.urdf",
                pos=self.env_cfg["base_init_pos"],
                quat=self.env_cfg["base_init_quat"],
            ),
        )

        # cameras must be added before the scene is built
        self.cams = [self.scene.add_camera(**cam_kwargs) for cam_kwargs in (cameras or [])]

        self.scene.build(n_envs=num_envs)

        # names to indices
        self.motors_dof_idx = torch.tensor(
            [self.robot.get_joint(name).dof_start for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_int,
            device=gs.device,
        )
        self.actions_dof_idx = torch.argsort(self.motors_dof_idx)

        # PD control parameters
        self.robot.set_dofs_kp([self.env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
        self.robot.set_dofs_kv([self.env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)

        # Define global gravity direction vector
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], dtype=gs.tc_float, device=gs.device)

        # Initial state
        self.init_base_pos = torch.tensor(self.env_cfg["base_init_pos"], dtype=gs.tc_float, device=gs.device)
        self.init_base_quat = torch.tensor(self.env_cfg["base_init_quat"], dtype=gs.tc_float, device=gs.device)
        self.inv_base_init_quat = inv_quat(self.init_base_quat)
        self.init_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][joint.name] for joint in self.robot.joints[1:]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.init_qpos = torch.concatenate((self.init_base_pos, self.init_base_quat, self.init_dof_pos))
        self.init_projected_gravity = transform_by_quat(self.global_gravity, self.inv_base_init_quat)

        # initialize buffers
        self.base_lin_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_ang_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.projected_gravity = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.rew_buf = torch.empty((self.num_envs,), dtype=gs.tc_float, device=gs.device)
        self.reset_buf = torch.ones((self.num_envs,), dtype=gs.tc_bool, device=gs.device)
        self.episode_length_buf = torch.empty((self.num_envs,), dtype=gs.tc_int, device=gs.device)
        self.commands = torch.empty((self.num_envs, self.num_commands), dtype=gs.tc_float, device=gs.device)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            device=gs.device,
            dtype=gs.tc_float,
        )
        self.commands_limits: tuple[torch.Tensor, torch.Tensor] = tuple(
            torch.tensor(values, dtype=gs.tc_float, device=gs.device)
            for values in zip(
                self.command_cfg["lin_vel_x_range"],
                self.command_cfg["lin_vel_y_range"],
                self.command_cfg["ang_vel_range"],
            )
        )
        self.actions = torch.zeros((self.num_envs, self.num_actions), dtype=gs.tc_float, device=gs.device)
        self.last_actions = torch.zeros_like(self.actions)
        self.dof_pos = torch.empty_like(self.actions)
        self.dof_vel = torch.empty_like(self.actions)
        self.last_dof_vel = torch.zeros_like(self.actions)
        self.base_pos = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_quat = torch.empty((self.num_envs, 4), dtype=gs.tc_float, device=gs.device)
        self.base_euler = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.default_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][name] for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.extras = dict()  # extra information for logging

        # prepare reward functions and multiply reward scales by dt
        self.reward_functions, self.episode_sums = dict(), dict()
        for name in self.reward_scales.keys():
            self.reward_scales[name] *= self.dt
            self.reward_functions[name] = getattr(self, "_reward_" + name)
            self.episode_sums[name] = torch.zeros((self.num_envs,), dtype=gs.tc_float, device=gs.device)

        self.reset()

    def set_commands(self, commands):
        """Overwrite the velocity commands [vx (m/s), vy (m/s), yaw rate (rad/s)] of all environments."""
        self.external_commands = True
        self.commands.copy_(torch.as_tensor(commands, dtype=gs.tc_float, device=gs.device).reshape(self.commands.shape))
        if hasattr(self, "obs_buf"):
            self._update_observation()  # the policy must see the new command

    def _resample_commands(self, envs_idx):
        if self.external_commands:
            return
        commands = gs_rand(*self.commands_limits, (self.num_envs,))
        # some environments get a pure straight / pure turning command, which appear in a figure-eight
        mode = torch.rand((self.num_envs,), device=gs.device)
        commands[:, 2] = torch.where(mode < self.command_cfg.get("straight_ratio", 0.0), 0.0, commands[:, 2])
        commands[:, 1] = torch.where(mode < self.command_cfg.get("no_lateral_ratio", 0.0), 0.0, commands[:, 1])
        if envs_idx is None:
            self.commands.copy_(commands)
        else:
            torch.where(envs_idx[:, None], commands, self.commands, out=self.commands)

    def step(self, actions):
        self.actions = torch.clip(actions, -self.env_cfg["clip_actions"], self.env_cfg["clip_actions"])
        exec_actions = self.last_actions if self.simulate_action_latency else self.actions
        target_dof_pos = exec_actions * self.env_cfg["action_scale"] + self.default_dof_pos
        self.robot.control_dofs_position(target_dof_pos[:, self.actions_dof_idx], slice(6, 18))
        self.scene.step()

        # update buffers
        self.episode_length_buf += 1
        self.base_pos = self.robot.get_pos()
        self.base_quat = self.robot.get_quat()
        self.base_euler = quat_to_xyz(
            transform_quat_by_quat(self.inv_base_init_quat, self.base_quat), rpy=True, degrees=True
        )
        inv_base_quat = inv_quat(self.base_quat)
        self.base_lin_vel = transform_by_quat(self.robot.get_vel(), inv_base_quat)
        self.base_ang_vel = transform_by_quat(self.robot.get_ang(), inv_base_quat)
        self.projected_gravity = transform_by_quat(self.global_gravity, inv_base_quat)
        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)
        self.dof_vel = self.robot.get_dofs_velocity(self.motors_dof_idx)

        # compute reward
        self.rew_buf.zero_()
        for name, reward_func in self.reward_functions.items():
            rew = reward_func() * self.reward_scales[name]
            self.rew_buf += rew
            self.episode_sums[name] += rew

        self._resample_commands(self.episode_length_buf % int(self.env_cfg["resampling_time_s"] / self.dt) == 0)

        # check termination and reset
        self.reset_buf = self.episode_length_buf > self.max_episode_length
        self.reset_buf |= torch.abs(self.base_euler[:, 1]) > self.env_cfg["termination_if_pitch_greater_than"]
        self.reset_buf |= torch.abs(self.base_euler[:, 0]) > self.env_cfg["termination_if_roll_greater_than"]
        self.reset_buf |= self.scene.rigid_solver.get_error_envs_mask()

        # Compute timeout
        self.extras["time_outs"] = (self.episode_length_buf > self.max_episode_length).to(dtype=gs.tc_float)

        # Reset environment if necessary
        self._reset_idx(self.reset_buf)

        # update observations
        self._update_observation()

        self.last_actions.copy_(self.actions)
        self.last_dof_vel.copy_(self.dof_vel)

        return self.get_observations(), self.rew_buf, self.reset_buf, self.extras

    def get_observations(self):
        return TensorDict({"policy": self.obs_buf}, batch_size=[self.num_envs])

    def _reset_idx(self, envs_idx=None):
        # reset state
        self.robot.set_qpos(self.init_qpos, envs_idx=envs_idx, zero_velocity=True, skip_forward=True)

        # reset buffers
        if envs_idx is None:
            self.base_pos.copy_(self.init_base_pos)
            self.base_quat.copy_(self.init_base_quat)
            self.projected_gravity.copy_(self.init_projected_gravity)
            self.dof_pos.copy_(self.init_dof_pos)
            self.base_lin_vel.zero_()
            self.base_ang_vel.zero_()
            self.dof_vel.zero_()
            self.actions.zero_()
            self.last_actions.zero_()
            self.last_dof_vel.zero_()
            self.episode_length_buf.zero_()
            self.reset_buf.fill_(True)
        else:
            torch.where(envs_idx[:, None], self.init_base_pos, self.base_pos, out=self.base_pos)
            torch.where(envs_idx[:, None], self.init_base_quat, self.base_quat, out=self.base_quat)
            torch.where(
                envs_idx[:, None], self.init_projected_gravity, self.projected_gravity, out=self.projected_gravity
            )
            torch.where(envs_idx[:, None], self.init_dof_pos, self.dof_pos, out=self.dof_pos)
            self.base_lin_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.base_ang_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.episode_length_buf.masked_fill_(envs_idx, 0)
            self.reset_buf.masked_fill_(envs_idx, True)

        # fill extras
        n_envs = envs_idx.sum() if envs_idx is not None else self.num_envs
        self.extras["episode"] = {}
        for key, value in self.episode_sums.items():
            if envs_idx is None:
                mean = value.mean()
            else:
                mean = torch.where(n_envs > 0, value[envs_idx].sum() / n_envs, 0.0)
            self.extras["episode"]["rew_" + key] = mean / self.env_cfg["episode_length_s"]
            if envs_idx is None:
                value.zero_()
            else:
                value.masked_fill_(envs_idx, 0.0)

        # random sample command upon reset
        self._resample_commands(envs_idx)

    def _update_observation(self):
        self.obs_buf = torch.concatenate(
            (
                self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
                self.projected_gravity,  # 3
                self.commands * self.commands_scale,  # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
                self.dof_vel * self.obs_scales["dof_vel"],  # 12
                self.actions,  # 12
            ),
            dim=-1,
        )

    def reset(self):
        self._reset_idx()
        self._update_observation()
        return self.get_observations()

    # Reward functions.
    def _reward_tracking_lin_vel(self):
        # Tracking of linear velocity commands (xy axes)
        lin_vel_error = torch.sum(torch.square(self.commands[:, :2] - self.base_lin_vel[:, :2]), dim=1)
        return torch.exp(-lin_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_tracking_ang_vel(self):
        # Tracking of angular velocity commands (yaw)
        ang_vel_error = torch.square(self.commands[:, 2] - self.base_ang_vel[:, 2])
        return torch.exp(-ang_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_lin_vel_z(self):
        # Penalize z axis base linear velocity
        return torch.square(self.base_lin_vel[:, 2])

    def _reward_action_rate(self):
        # Penalize changes in actions
        return torch.sum(torch.square(self.last_actions - self.actions), dim=1)

    def _reward_similar_to_default(self):
        # Penalize joint poses far away from default pose
        return torch.sum(torch.abs(self.dof_pos - self.default_dof_pos), dim=1)

    def _reward_ang_vel_xy(self):
        # Penalize roll / pitch angular velocity (keeps the body flat while turning)
        return torch.sum(torch.square(self.base_ang_vel[:, :2]), dim=1)

    def _reward_orientation(self):
        # Penalize non-flat base orientation
        return torch.sum(torch.square(self.projected_gravity[:, :2]), dim=1)

    def _reward_dof_acc(self):
        # Penalize joint accelerations (smoother gait)
        return torch.sum(torch.square((self.last_dof_vel - self.dof_vel) / self.dt), dim=1)

    def _reward_base_height(self):
        # Penalize base height away from target
        return torch.square(self.base_pos[:, 2] - self.reward_cfg["base_height_target"])

### 学習（fig8_train.py）
公式は「前進 0.5 m/s のみ」。ここでは前進 0〜1.0 m/s・横 ±0.3 m/s・旋回 ±1.0 rad/s をランダムに与え、どの方向にも曲がれる方策を学習します。

In [ ]:
%%writefile fig8_train.py
import argparse
import os
import pickle
import shutil
from importlib import metadata

try:
    if int(metadata.version("rsl-rl-lib").split(".")[0]) < 5:
        raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please install 'rsl-rl-lib>=5.0.0'.") from e
from rsl_rl.runners import OnPolicyRunner

import genesis as gs

from fig8_env import Fig8Env


def get_train_cfg(exp_name):
    train_cfg_dict = {
        "algorithm": {
            "class_name": "PPO",
            "clip_param": 0.2,
            "desired_kl": 0.01,
            "entropy_coef": 0.01,
            "gamma": 0.99,
            "lam": 0.95,
            "learning_rate": 0.001,
            "max_grad_norm": 1.0,
            "num_learning_epochs": 5,
            "num_mini_batches": 4,
            "schedule": "adaptive",
            "use_clipped_value_loss": True,
            "value_loss_coef": 1.0,
        },
        "actor": {
            "class_name": "MLPModel",
            "hidden_dims": [512, 256, 128],
            "activation": "elu",
            "distribution_cfg": {
                "class_name": "GaussianDistribution",
                "init_std": 1.0,
                "std_type": "scalar",
            },
        },
        "critic": {
            "class_name": "MLPModel",
            "hidden_dims": [512, 256, 128],
            "activation": "elu",
        },
        "obs_groups": {
            "actor": ["policy"],
            "critic": ["policy"],
        },
        "num_steps_per_env": 24,
        "save_interval": 100,
        "run_name": exp_name,
        "logger": "tensorboard",
    }

    return train_cfg_dict


def get_cfgs():
    env_cfg = {
        "num_actions": 12,
        # joint/link names
        "default_joint_angles": {  # [rad]
            "FL_hip_joint": 0.0,
            "FR_hip_joint": 0.0,
            "RL_hip_joint": 0.0,
            "RR_hip_joint": 0.0,
            "FL_thigh_joint": 0.8,
            "FR_thigh_joint": 0.8,
            "RL_thigh_joint": 1.0,
            "RR_thigh_joint": 1.0,
            "FL_calf_joint": -1.5,
            "FR_calf_joint": -1.5,
            "RL_calf_joint": -1.5,
            "RR_calf_joint": -1.5,
        },
        "joint_names": [
            "FR_hip_joint",
            "FR_thigh_joint",
            "FR_calf_joint",
            "FL_hip_joint",
            "FL_thigh_joint",
            "FL_calf_joint",
            "RR_hip_joint",
            "RR_thigh_joint",
            "RR_calf_joint",
            "RL_hip_joint",
            "RL_thigh_joint",
            "RL_calf_joint",
        ],
        # PD
        "kp": 20.0,
        "kd": 0.5,
        # termination
        "termination_if_roll_greater_than": 10,  # degree
        "termination_if_pitch_greater_than": 10,
        # base pose
        "base_init_pos": [0.0, 0.0, 0.42],
        "base_init_quat": [1.0, 0.0, 0.0, 0.0],
        "episode_length_s": 20.0,
        "resampling_time_s": 4.0,
        "action_scale": 0.25,
        "simulate_action_latency": True,
        "clip_actions": 100.0,
    }
    obs_cfg = {
        "obs_scales": {
            "lin_vel": 2.0,
            "ang_vel": 0.25,
            "dof_pos": 1.0,
            "dof_vel": 0.05,
        },
    }
    reward_cfg = {
        "tracking_sigma": 0.25,
        "base_height_target": 0.3,
        "feet_height_target": 0.075,
        "reward_scales": {
            "tracking_lin_vel": 1.0,
            "tracking_ang_vel": 0.5,  # 0.2 in the original: turning matters for a figure-eight
            "lin_vel_z": -1.0,
            "base_height": -50.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
            "ang_vel_xy": -0.05,
            "orientation": -1.0,
            "dof_acc": -2.5e-7,
        },
    }
    command_cfg = {
        "num_commands": 3,
        # original example: forward 0.5 m/s only. Here: forward speed + yaw rate (+ a little lateral)
        "lin_vel_x_range": [0.0, 1.0],
        "lin_vel_y_range": [-0.3, 0.3],
        "ang_vel_range": [-1.0, 1.0],
        "straight_ratio": 0.2,  # ratio of envs with yaw rate 0
        "no_lateral_ratio": 0.5,  # ratio of envs with lateral velocity 0
    }

    return env_cfg, obs_cfg, reward_cfg, command_cfg


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "-e", "--exp-name", type=str, default="go2-fig8", help="Experiment name; also the log directory under logs/"
    )
    parser.add_argument("-b", "--num-envs", type=int, default=4096, help="Number of parallel environments")
    parser.add_argument("--max-iterations", type=int, default=501, help="Number of learning iterations")
    parser.add_argument("--log-root", type=str, default="logs", help="Root directory of the logs (e.g. on Google Drive)")
    parser.add_argument("--resume", type=str, default=None, help="Checkpoint (.pt) to resume training from")
    parser.add_argument("--cpu", action="store_true", help="Run on CPU (only for a quick smoke test)")
    parser.add_argument("--seed", type=int, default=1, help="Random seed")
    args = parser.parse_args()

    log_dir = os.path.join(args.log_root, args.exp_name)
    env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
    train_cfg = get_train_cfg(args.exp_name)

    if os.path.exists(log_dir) and args.resume is None:
        shutil.rmtree(log_dir)
    os.makedirs(log_dir, exist_ok=True)

    with open(f"{log_dir}/cfgs.pkl", "wb") as f:
        pickle.dump([env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg], f)

    # Training throughput comes from the 4096 parallel environments, which need a GPU.
    gs.init(backend=gs.cpu if args.cpu else gs.gpu, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)

    env = Fig8Env(
        num_envs=args.num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg, command_cfg=command_cfg
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    if args.resume is not None:
        runner.load(args.resume)

    runner.learn(num_learning_iterations=args.max_iterations, init_at_random_ep_len=True)


if __name__ == "__main__":
    main()

"""
# training
python fig8_train.py -e go2-fig8 -b 4096 --max-iterations 501
"""

### 八の字評価（fig8_eval.py）
レムニスケート経路を Pure Pursuit で追従。俯瞰カメラと追従カメラで録画し、軌跡と指標（周回数、経路からのずれ等）を保存します。

In [ ]:
%%writefile fig8_eval.py
"""
Figure-eight walking with a trained Go2 locomotion policy.

Hierarchical control:
  - high level : pure-pursuit path follower on a figure-eight (lemniscate of Bernoulli)
                 -> velocity command [vx, vy, yaw rate]
  - low level  : RL policy (PPO, trained by fig8_train.py) -> 12 joint position targets

The run is recorded headlessly by two cameras (overview + follow) and the trajectory is
saved with tracking metrics, so it works on Google Colab without a display.

# evaluation
python fig8_eval.py -e go2-fig8 --ckpt 500
"""

import argparse
import json
import math
import os
import pickle
from importlib import metadata

import numpy as np
import torch

try:
    if int(metadata.version("rsl-rl-lib").split(".")[0]) < 5:
        raise ImportError
except (metadata.PackageNotFoundError, ImportError, ValueError) as e:
    raise ImportError("Please install 'rsl-rl-lib>=5.0.0'.") from e
from rsl_rl.runners import OnPolicyRunner

import genesis as gs
from genesis.utils.geom import quat_to_xyz

from fig8_env import Fig8Env


def figure_eight(a, n=4000):
    """Lemniscate of Bernoulli with half-width a [m], starting at (a, 0) heading +y."""
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    s = np.sin(t)
    d = 1.0 + s**2
    return np.stack((a * np.cos(t) / d, a * s * np.cos(t) / d), axis=-1)


class PurePursuit:
    """Pure-pursuit follower on a closed path that turns a pose into a velocity command."""

    def __init__(self, path, speed, lookahead, max_yaw_rate):
        self.path = path
        self.speed = speed
        self.lookahead = lookahead
        self.max_yaw_rate = max_yaw_rate
        seg = np.linalg.norm(np.roll(path, -1, axis=0) - path, axis=1)
        self.ds = seg.mean()
        self.length = seg.sum()
        self.idx = 0  # index of the nearest path point
        self.progress = 0  # number of path points passed (for lap counting)

    def __call__(self, pos_xy, yaw):
        n = len(self.path)
        # search the nearest point only a little ahead of the previous one, so the follower
        # does not jump to the other branch at the crossing of the eight
        window = (self.idx + np.arange(-20, 200)) % n
        d = np.linalg.norm(self.path[window] - pos_xy, axis=1)
        k = int(np.argmin(d))
        new_idx = int(window[k])
        self.progress += (new_idx - self.idx) % n if (new_idx - self.idx) % n < n // 2 else 0
        self.idx = new_idx
        cross_track_error = float(d[k])

        target = self.path[(self.idx + int(self.lookahead / self.ds)) % n]
        dx, dy = target - pos_xy
        # angle of the target in the robot frame
        alpha = math.atan2(dy, dx) - yaw
        alpha = math.atan2(math.sin(alpha), math.cos(alpha))
        # pure pursuit: curvature = 2 sin(alpha) / L
        yaw_rate = 2.0 * self.speed * math.sin(alpha) / self.lookahead
        yaw_rate = float(np.clip(yaw_rate, -self.max_yaw_rate, self.max_yaw_rate))
        # slow down when the heading error is large
        vx = self.speed * max(0.3, math.cos(alpha))
        return np.array([vx, 0.0, yaw_rate]), cross_track_error

    @property
    def laps(self):
        return self.progress / len(self.path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-fig8")
    parser.add_argument("--ckpt", type=int, default=500, help="Checkpoint iteration to load")
    parser.add_argument("--log-root", type=str, default="logs")
    parser.add_argument("--out-dir", type=str, default="outputs")
    parser.add_argument("--size", type=float, default=3.0, help="Half-width of the figure-eight [m]")
    parser.add_argument("--speed", type=float, default=0.6, help="Walking speed command [m/s]")
    parser.add_argument("--lookahead", type=float, default=0.6, help="Pure-pursuit look-ahead distance [m]")
    parser.add_argument("--max-yaw-rate", type=float, default=1.0, help="Must stay inside the trained range")
    parser.add_argument("--laps", type=float, default=3.0, help="Number of laps to walk")
    parser.add_argument("--max-time", type=float, default=170.0, help="Time limit [s] (video must be <= 3 min)")
    parser.add_argument("--fps", type=int, default=25)
    parser.add_argument("--res", type=int, nargs=2, default=(960, 540))
    parser.add_argument("--no-video", action="store_true")
    parser.add_argument("--cpu", action="store_true")
    args = parser.parse_args()

    gs.init(backend=gs.cpu if args.cpu else gs.gpu, logging_level="warning")

    log_dir = os.path.join(args.log_root, args.exp_name)
    with open(os.path.join(log_dir, "cfgs.pkl"), "rb") as f:
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(f)
    reward_cfg["reward_scales"] = {}
    # start on the right tip of the eight, heading +y (tangent of the path)
    env_cfg["base_init_pos"] = [args.size, 0.0, 0.42]
    env_cfg["base_init_quat"] = [math.cos(math.pi / 4), 0.0, 0.0, math.sin(math.pi / 4)]
    env_cfg["episode_length_s"] = args.max_time + 10.0

    a = args.size
    cameras = []
    if not args.no_video:
        cameras = [
            # overview camera looking at the whole eight
            # (debug=True so that the reference path and trail drawn with draw_debug_* appear in the video)
            dict(res=tuple(args.res), pos=(0.0, -1.3 * a, 1.2 * a), lookat=(0.0, -0.1 * a, 0.0), fov=55, debug=True),
            # follow camera, moved with the robot
            dict(res=tuple(args.res), pos=(a + 1.8, -1.2, 0.9), lookat=(a, 0.0, 0.3), fov=50, debug=True),
        ]
    env = Fig8Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
        cameras=cameras,
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{args.ckpt}.pt"), map_location=str(gs.device))
    policy = runner.get_inference_policy(device=gs.device)

    path = figure_eight(a)
    follower = PurePursuit(path, args.speed, args.lookahead, args.max_yaw_rate)

    os.makedirs(args.out_dir, exist_ok=True)
    if cameras:
        # draw the reference path on the ground
        env.scene.draw_debug_trajectory(
            np.column_stack((np.concatenate((path[::20], path[:1])), np.full(len(path[::20]) + 1, 0.005))),
            radius=0.015,
            color=(1.0, 0.5, 0.0, 1.0),
        )
        env.cams[1].follow_entity(env.robot, smoothing=0.9)
        env.cams[0].start_recording(os.path.join(args.out_dir, "overview.mp4"), fps=args.fps)
        env.cams[1].start_recording(os.path.join(args.out_dir, "follow.mp4"), fps=args.fps)

    command = np.zeros(3)
    env.set_commands(command)
    obs = env.reset()
    log = {"t": [], "pos": [], "yaw": [], "cmd": [], "lin_vel": [], "ang_vel": [], "cte": []}
    n_falls = 0
    max_steps = int(args.max_time / env.dt)
    with torch.no_grad():
        for step in range(max_steps):
            pos = env.base_pos[0].cpu().numpy()
            yaw = float(quat_to_xyz(env.base_quat, rpy=True)[0, 2])
            command, cte = follower(pos[:2], yaw)
            env.set_commands(command)
            obs = env.get_observations()
            actions = policy(obs)
            obs, _, dones, _ = env.step(actions)
            if bool(dones[0]) and step < max_steps - 1:
                n_falls += 1
                follower.idx = 0  # the robot is reset to the start point

            log["t"].append(step * env.dt)
            log["pos"].append(pos)
            log["yaw"].append(yaw)
            log["cmd"].append(command)
            log["lin_vel"].append(env.base_lin_vel[0].cpu().numpy())
            log["ang_vel"].append(env.base_ang_vel[0].cpu().numpy())
            log["cte"].append(cte)
            if cameras and step % 25 == 0:
                # leave a trail of the robot's actual path
                env.scene.draw_debug_sphere((pos[0], pos[1], 0.02), radius=0.03, color=(0.1, 0.4, 1.0, 1.0))
            if follower.laps >= args.laps:
                break

    if cameras:
        for cam in env.cams:
            cam.stop_recording()

    log = {k: np.asarray(v) for k, v in log.items()}
    np.savez(os.path.join(args.out_dir, "trajectory.npz"), path=path, **log)
    cte = log["cte"]
    metrics = {
        "time_s": float(log["t"][-1]),
        "laps": round(follower.laps, 2),
        "path_length_m": round(follower.length, 2),
        "mean_speed_mps": round(float(np.linalg.norm(log["lin_vel"][:, :2], axis=1).mean()), 3),
        "cross_track_error_rms_m": round(float(np.sqrt(np.mean(cte**2))), 3),
        "cross_track_error_max_m": round(float(cte.max()), 3),
        "yaw_rate_tracking_rmse": round(float(np.sqrt(np.mean((log["cmd"][:, 2] - log["ang_vel"][:, 2]) ** 2))), 3),
        "falls": n_falls,
    }
    with open(os.path.join(args.out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile plot_results.py
"""
Plot the figure-eight run saved by fig8_eval.py (trajectory, commands, tracking error).

python plot_results.py --out-dir outputs
"""

import argparse
import json
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", type=str, default="outputs")
    args = parser.parse_args()

    d = np.load(os.path.join(args.out_dir, "trajectory.npz"))
    with open(os.path.join(args.out_dir, "metrics.json")) as f:
        metrics = json.load(f)
    path, pos, t = d["path"], d["pos"], d["t"]

    fig = plt.figure(figsize=(14, 6))
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(*np.vstack((path, path[:1])).T, "--", color="tab:orange", lw=2, label="reference (figure-eight)")
    sc = ax.scatter(pos[:, 0], pos[:, 1], c=t, s=2, cmap="viridis", label="robot base")
    ax.plot(*pos[0, :2], "ko", label="start")
    fig.colorbar(sc, ax=ax, label="time [s]", shrink=0.5)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(
        f"Trajectory: {metrics['laps']} laps, cross-track RMS {metrics['cross_track_error_rms_m'] * 100:.1f} cm"
    )
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(3, 2, 2)
    ax.plot(t, d["lin_vel"][:, 0], color="tab:orange", alpha=0.6, label="actual vx")
    ax.plot(t, d["cmd"][:, 0], color="k", lw=1.5, label="command vx")
    ax.set_ylabel("[m/s]")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
    ax = fig.add_subplot(3, 2, 4)
    ax.plot(t, d["ang_vel"][:, 2], color="tab:orange", alpha=0.6, label="actual yaw rate")
    ax.plot(t, d["cmd"][:, 2], color="k", lw=1.5, label="command yaw rate")
    ax.set_ylabel("[rad/s]")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
    ax = fig.add_subplot(3, 2, 6)
    ax.plot(t, d["cte"] * 100, color="tab:red")
    ax.set_ylabel("cross-track\nerror [cm]")
    ax.set_xlabel("time [s]")
    ax.grid(alpha=0.3)

    fig.tight_layout()
    out = os.path.join(args.out_dir, "fig8_result.png")
    fig.savefig(out, dpi=120)
    print("saved", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_video.py
"""
Build the submission video from the two camera recordings of fig8_eval.py:
title card (with the playback speed) -> overview | follow side by side -> result plot.

python make_video.py --out-dir outputs --speed 1
"""

import argparse
import os
import subprocess

from PIL import Image, ImageDraw, ImageFont

W, H = 1920, 540


def card(path, lines, size=(W, H)):
    img = Image.new("RGB", size, (16, 24, 40))
    draw = ImageDraw.Draw(img)
    y = (size[1] - sum(int(font_size * 1.6) for _, font_size in lines)) / 2
    for text, font_size in lines:
        font = ImageFont.load_default(size=font_size)
        w = draw.textlength(text, font=font)
        draw.text(((size[0] - w) / 2, y), text, fill=(240, 240, 240), font=font)
        y += int(font_size * 1.6)
    img.save(path)


def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", type=str, default="outputs")
    parser.add_argument("--speed", type=float, default=1.0, help="Playback speed of the main part (1 or 2)")
    parser.add_argument("--title", type=str, default="Figure-eight walking of a quadruped robot (Go2)")
    args = parser.parse_args()
    o = lambda name: os.path.join(args.out_dir, name)  # noqa: E731

    speed_text = "1x speed (real time)" if args.speed == 1 else f"{args.speed:g}x speed"
    card(
        o("title.png"),
        [
            (args.title, 64),
            ("Genesis simulator / RL locomotion policy (PPO) + pure-pursuit path following", 36),
            ("left: overview camera   right: follow camera   orange: reference path   blue: robot trail", 30),
            (speed_text, 44),
        ],
    )

    # 3 s title card
    run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", o("title.png"), "-t", "3", "-r", "25",
         "-pix_fmt", "yuv420p", "-c:v", "libx264", o("part_title.mp4")])
    # side by side main part (+ speed up)
    run(["ffmpeg", "-y", "-v", "error", "-i", o("overview.mp4"), "-i", o("follow.mp4"), "-filter_complex",
         f"[0:v]scale=960:540[a];[1:v]scale=960:540[b];[a][b]hstack=inputs=2,setpts=PTS/{args.speed},fps=25[v]",
         "-map", "[v]", "-pix_fmt", "yuv420p", "-c:v", "libx264", o("part_main.mp4")])
    parts = [o("part_title.mp4"), o("part_main.mp4")]
    # 5 s result plot
    if os.path.exists(o("fig8_result.png")):
        run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", o("fig8_result.png"), "-t", "5", "-r", "25",
             "-vf", f"scale={W}:{H}:force_original_aspect_ratio=decrease,pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:white",
             "-pix_fmt", "yuv420p", "-c:v", "libx264", o("part_result.mp4")])
        parts.append(o("part_result.mp4"))

    with open(o("parts.txt"), "w") as f:
        f.writelines(f"file '{os.path.abspath(p)}'\n" for p in parts)
    run(["ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0", "-i", o("parts.txt"), "-c", "copy",
         o("fig8_submission.mp4")])
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                          o("fig8_submission.mp4")], capture_output=True, text=True).stdout.strip()
    print(f"saved {o('fig8_submission.mp4')} ({float(out):.1f} s; the assignment requires 60-180 s)")


if __name__ == "__main__":
    main()

## Step 2. 強化学習
学習曲線は TensorBoard で確認できます（任意）。`Mean reward` が上がり、`rew_tracking_lin_vel`/`rew_tracking_ang_vel` が伸びていれば順調です。

途中で切断された場合は `--resume logs/go2-fig8/model_300.pt` のように続きから学習できます。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
!python fig8_train.py -e go2-fig8 -b 4096 --max-iterations 501

## Step 3. 八の字歩行の評価＋録画
`--speed`（歩行速度）、`--size`（八の字の半幅[m]）、`--laps`（周回数）を変えて試せます。動画が1〜3分に収まるよう、デフォルトは 0.6 m/s・半幅3 m・3周（約80秒）です。

In [ ]:
!python fig8_eval.py -e go2-fig8 --ckpt 500 --speed 0.6 --size 3.0 --laps 3 --out-dir outputs

## Step 4. 結果グラフと提出用動画

In [ ]:
!python plot_results.py --out-dir outputs
from IPython.display import Image, display
display(Image("outputs/fig8_result.png"))

In [ ]:
# 動画が 60 秒未満なら --laps を増やす / 180 秒を超えるなら --speed 2 で2倍速（タイトルに自動で明記）
!python make_video.py --out-dir outputs --speed 1

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open("outputs/fig8_submission.mp4", "rb").read()
HTML(f'<video width=960 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 提出物
- 動画：Drive の `pai_fig8/outputs/fig8_submission.mp4` を YouTube にアップロード（タイトル `PAI最終課題_omnicampusアカウント名`、限定公開以上、子ども向け「いいえ」）
- コード：この Notebook と `*.py` を GitHub の public リポジトリへ
- レポート：`outputs/metrics.json` と `outputs/fig8_result.png` の結果を記入して PDF 化